In [13]:
import os
import requests
from PIL import Image
from io import BytesIO

import pandas as pd
import matplotlib.pyplot as plt

In [14]:
# NASA GIBS WMS endpoint
BASE_URL = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"

# Output folders
IMAGE_DIR = "modis_sst_images"
os.makedirs(IMAGE_DIR, exist_ok=True)

# Image size for downloaded maps
WIDTH = 800
HEIGHT = 800

In [15]:
MODIS_LAYERS = {
    "Aqua_Day": "MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly",
    "Aqua_Night": "MODIS_Aqua_L3_SST_Thermal_4km_Night_Monthly",
    "Terra_Day": "MODIS_Terra_L3_SST_Thermal_4km_Day_Monthly",
    "Terra_Night": "MODIS_Terra_L3_SST_Thermal_4km_Night_Monthly"
}

MODIS_LAYERS

{'Aqua_Day': 'MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly',
 'Aqua_Night': 'MODIS_Aqua_L3_SST_Thermal_4km_Night_Monthly',
 'Terra_Day': 'MODIS_Terra_L3_SST_Thermal_4km_Day_Monthly',
 'Terra_Night': 'MODIS_Terra_L3_SST_Thermal_4km_Night_Monthly'}

In [16]:
REGIONS = {
    "California Coast": {
        "bbox": "25,-130,45,-110",
        "description": "Northeast Pacific along California and Baja California"
    },
    "Gulf of Mexico": {
        "bbox": "18,-100,32,-80",
        "description": "Warm semi-enclosed basin near the southern United States"
    },
    "East Coast": {
        "bbox": "25,-85,45,-65",
        "description": "Western Atlantic along the U.S. East Coast"
    }
}

REGIONS

{'California Coast': {'bbox': '25,-130,45,-110',
  'description': 'Northeast Pacific along California and Baja California'},
 'Gulf of Mexico': {'bbox': '18,-100,32,-80',
  'description': 'Warm semi-enclosed basin near the southern United States'},
 'East Coast': {'bbox': '25,-85,45,-65',
  'description': 'Western Atlantic along the U.S. East Coast'}}

In [17]:
def make_filename(region, satellite, time_of_day, date):
    """
    Creates a clean filename for a MODIS SST image.
    """
    region_clean = region.lower().replace(" ", "_")
    satellite_clean = satellite.lower()
    time_clean = time_of_day.lower()
    date_clean = date.replace("-", "_")
    
    return f"{region_clean}_{satellite_clean}_{time_clean}_{date_clean}.png"



def download_modis_sst_image(region, satellite, time_of_day, date):
    """
    Downloads one MODIS SST image from NASA GIBS WMS API.

    Parameters
    ----------
    region : str
        Region name from REGIONS.
    satellite : str
        Either "Aqua" or "Terra".
    time_of_day : str
        Either "Day" or "Night".
    date : str
        Date in YYYY-MM-DD format.

    Returns
    -------
    dict
        Metadata about the downloaded image.
    """
    
    layer_key = f"{satellite}_{time_of_day}"
    layer_name = MODIS_LAYERS[layer_key]
    bbox = REGIONS[region]["bbox"]
    filename = make_filename(region, satellite, time_of_day, date)
    filepath = os.path.join(IMAGE_DIR, filename)
    
    params = {
        "SERVICE": "WMS",
        "REQUEST": "GetMap",
        "VERSION": "1.3.0",
        "LAYERS": layer_name,
        "STYLES": "",
        "CRS": "EPSG:4326",
        "BBOX": bbox,
        "WIDTH": str(WIDTH),
        "HEIGHT": str(HEIGHT),
        "FORMAT": "image/png",
        "TRANSPARENT": "false",
        "TIME": date
    }
    
    response = requests.get(BASE_URL, params=params)
    content_type = response.headers.get("Content-Type", "")
    
    print(date, region, satellite, time_of_day, response.status_code, content_type)
    
    if "image" in content_type:
        with open(filepath, "wb") as f:
            f.write(response.content)
            
        status = "success"
        error_message = ""
    else:
        filepath = None
        status = "failed"
        error_message = response.text[:500]
        print("NASA returned an error:")
        print(error_message)
    
    return {
        "date": date,
        "year": int(date[:4]),
        "month": int(date[5:7]),
        "region": region,
        "bbox": bbox,
        "satellite": satellite,
        "time_of_day": time_of_day,
        "layer": layer_name,
        "image_path": filepath,
        "status": status,
        "error_message": error_message
    }

In [18]:
all_metadata = []

years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]
months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
regions = list(REGIONS.keys())
satellites = ["Aqua", "Terra"]
times_of_day = ["Day", "Night"]

for region in regions:
    for satellite in satellites:
        for time_of_day in times_of_day:
            for year in years:
                for month in months:
                    date = f"{year}-{month:02d}-01"
                    
                    meta = download_modis_sst_image(
                        region=region,
                        satellite=satellite,
                        time_of_day=time_of_day,
                        date=date
                    )
                    
                    all_metadata.append(meta)

full_metadata_df = pd.DataFrame(all_metadata)
full_metadata_df.head()

2010-01-01 California Coast Aqua Day 200 image/png
2010-02-01 California Coast Aqua Day 200 image/png
2010-03-01 California Coast Aqua Day 200 image/png
2010-04-01 California Coast Aqua Day 200 image/png
2010-05-01 California Coast Aqua Day 200 image/png
2010-06-01 California Coast Aqua Day 200 image/png
2010-07-01 California Coast Aqua Day 200 image/png
2010-08-01 California Coast Aqua Day 200 image/png
2010-09-01 California Coast Aqua Day 200 image/png
2010-10-01 California Coast Aqua Day 200 image/png
2010-11-01 California Coast Aqua Day 200 image/png
2010-12-01 California Coast Aqua Day 200 image/png
2011-01-01 California Coast Aqua Day 200 image/png
2011-02-01 California Coast Aqua Day 200 image/png
2011-03-01 California Coast Aqua Day 200 image/png
2011-04-01 California Coast Aqua Day 200 image/png
2011-05-01 California Coast Aqua Day 200 image/png
2011-06-01 California Coast Aqua Day 200 image/png
2011-07-01 California Coast Aqua Day 200 image/png
2011-08-01 California Coast Aqu

,date,year,month,region,bbox,satellite,time_of_day,layer,image_path,status,error_message
0,2010-01-01,2010,1,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
1,2010-02-01,2010,2,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
2,2010-03-01,2010,3,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
3,2010-04-01,2010,4,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
4,2010-05-01,2010,5,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,


In [19]:
successful_metadata_df = full_metadata_df[full_metadata_df["status"] == "success"].copy()

successful_metadata_df.to_csv("modis_sst_metadata.csv", index=False)

print("Saved modis_sst_metadata.csv")
print("Number of successful images:", len(successful_metadata_df))
successful_metadata_df.head()

Saved modis_sst_metadata.csv
Number of successful images: 2015


,date,year,month,region,bbox,satellite,time_of_day,layer,image_path,status,error_message
0,2010-01-01,2010,1,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
1,2010-02-01,2010,2,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
2,2010-03-01,2010,3,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
3,2010-04-01,2010,4,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,
4,2010-05-01,2010,5,California Coast,"25,-130,45,-110",Aqua,Day,MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly,modis_sst_images/california_coast_aqua_day_201...,success,


In [22]:
import pandas as pd

df = pd.read_csv("modis_sst_metadata.csv")

df["image_path"] = df["image_path"].astype(str)

df["image_path"] = df["image_path"].apply(
    lambda path: path.replace("\\", "/").split("modis_sst_images/")[-1]
)

df["image_path"] = "modis_sst_images/" + df["image_path"]

df.to_csv("modis_sst_metadata.csv", index=False)

df.columns

Index(['date', 'year', 'month', 'region', 'bbox', 'satellite', 'time_of_day',
       'layer', 'image_path', 'status', 'error_message'],
      dtype='str')

In [23]:
from PIL import Image
import numpy as np
import pandas as pd
import os

metadata_df = pd.read_csv("modis_sst_metadata.csv")

summary_rows = []

def analyze_sst_image(image_path):

    img = Image.open(image_path).convert("RGB")

    arr = np.array(img).astype(np.float32)

    red = arr[:, :, 0]
    green = arr[:, :, 1]
    blue = arr[:, :, 2]

    # Warmth index
    warmth = (red - blue) / 255.0

    # Remove dark background pixels
    mask = (red + green + blue) > 30

    valid = warmth[mask]

    if len(valid) == 0:
        return None

    return {
        "avg_warmth_score": float(valid.mean()),
        "max_warmth_score": float(valid.max()),
        "warm_area_percent": float((valid > 0.15).mean() * 100),
        "valid_pixel_count": int(len(valid))
    }

for _, row in metadata_df.iterrows():

    path = row["image_path"]

    if not os.path.exists(path):
        print("Missing:", path)
        continue

    stats = analyze_sst_image(path)

    if stats is None:
        continue

    summary_rows.append({
        **row.to_dict(),
        **stats
    })

summary_df = pd.DataFrame(summary_rows)

# MONTHLY BASELINE

baseline_df = (
    summary_df
    .groupby([
        "region",
        "satellite",
        "time_of_day",
        "month"
    ])["avg_warmth_score"]
    .mean()
    .reset_index()
    .rename(columns={
        "avg_warmth_score": "monthly_baseline"
    })
)

summary_df = summary_df.merge(
    baseline_df,
    on=["region", "satellite", "time_of_day", "month"],
    how="left"
)

# ANOMALY

summary_df["anomaly_score"] = (
    summary_df["avg_warmth_score"]
    - summary_df["monthly_baseline"]
)

# WARMTH RANK

summary_df["warmth_rank"] = (
    summary_df
    .groupby([
        "region",
        "satellite",
        "time_of_day",
        "month"
    ])["anomaly_score"]
    .rank(
        ascending=False,
        method="dense"
    )
    .astype(int)
)

# COLUMN ORDER

summary_df = summary_df[[
    "date",
    "year",
    "month",
    "region",
    "satellite",
    "time_of_day",
    "layer",
    "image_path",
    "avg_warmth_score",
    "monthly_baseline",
    "anomaly_score",
    "max_warmth_score",
    "warm_area_percent",
    "valid_pixel_count",
    "warmth_rank"
]]

summary_df.to_csv(
    "modis_anomaly_summary.csv",
    index=False
)

print("Saved modis_anomaly_summary.csv")
print(summary_df.head())

Saved modis_anomaly_summary.csv
         date  year  month            region satellite time_of_day  \
0  2010-01-01  2010      1  California Coast      Aqua         Day   
1  2010-02-01  2010      2  California Coast      Aqua         Day   
2  2010-03-01  2010      3  California Coast      Aqua         Day   
3  2010-04-01  2010      4  California Coast      Aqua         Day   
4  2010-05-01  2010      5  California Coast      Aqua         Day   

                                       layer  \
0  MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly   
1  MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly   
2  MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly   
3  MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly   
4  MODIS_Aqua_L3_SST_Thermal_4km_Day_Monthly   

                                          image_path  avg_warmth_score  \
0  modis_sst_images/california_coast_aqua_day_201...         -0.348802   
1  modis_sst_images/california_coast_aqua_day_201...         -0.423939   
2  modis_sst_images/california_coast